In [1]:
# Install necessary packages
!pip install pandas
!pip install google_play_scraper
!pip install Sastrawi
!pip install yake
!pip install vaderSentiment

import pandas as pd
from google_play_scraper import reviews, Sort
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import yake
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from IPython.display import display

# Fungsi untuk mengambil review dari Google Play Store
def fetch_reviews(app_id, count=1000, country='id', lang='id'):
    apps_review, _ = reviews(app_id, country=country, lang=lang, count=count, filter_score_with=None, sort=Sort.NEWEST)
    df = pd.DataFrame(apps_review)
    return df

# Fungsi case folding
def case_folding(text):
    return text.lower()

# Fungsi tokenisasi
def tokenize(text):
    words = re.findall(r'\b\w+\b', text)
    return words

# Fungsi stemming menggunakan Sastrawi
def stem_text(text):
    stemmer_factory = StemmerFactory()
    stemmer = stemmer_factory.create_stemmer()
    return stemmer.stem(text)

# Fungsi menghapus stopword menggunakan Sastrawi
def remove_stopwords(text):
    stopword_remover_factory = StopWordRemoverFactory()
    stopword_remover = stopword_remover_factory.create_stop_word_remover()
    return stopword_remover.remove(text)

# Fungsi untuk memproses teks secara lengkap
def process_text(text):
    text_lower = case_folding(text)
    tokens = tokenize(text_lower)
    stemmed_text = stem_text(' '.join(tokens))
    text_without_stopwords = remove_stopwords(stemmed_text)
    return text_without_stopwords

# Fungsi untuk memproses kolom pada DataFrame
def process_column(dataframe, column_name):
    dataframe[column_name] = dataframe[column_name].apply(process_text)

# Fungsi untuk analisis sentimen dengan VADER
def sentiment_analysis_vader(text):
    analyzer = SentimentIntensityAnalyzer()
    sentiment_dict = analyzer.polarity_scores(text)
    return sentiment_dict['compound']

# Fungsi untuk mengubah nilai sentimen menjadi skala Likert
def sentiment_to_likert(score, scale=5):
    if scale == 5:
        if score >= 0.5:
            return 5
        elif score >= 0.1:
            return 4
        elif score > -0.1:
            return 3
        elif score > -0.5:
            return 2
        else:
            return 1
    elif scale == 3:
        if score >= 0.2:
            return 3
        elif score > -0.2:
            return 2
        else:
            return 1

# Fungsi untuk mencocokkan review dengan kata kunci tertentu
def find_similar_reviews(reviews, keywords_similarity, threshold=0.05):
    similar_reviews = []
    for i, review in enumerate(reviews, start=1):
        similarity_score = 0
        for keyword_list in keywords_similarity.values():
            for keyword in keyword_list:
                if keyword in review:
                    similarity_score += 1
                    break
        if len(review.split()) > 0 and similarity_score / len(review.split()) >= threshold:
            similar_reviews.append((i, review, similarity_score / len(review.split())))
    return similar_reviews

# Kata kunci untuk similarity
keywords_similarity = {
    'Mudah': ['mudah', 'sederhana', 'simpel', 'simple', 'lancar', 'easy', 'accessible', 'aksesibilitas', 'keterjangkauan', 'mdh', 'ringan', 'gampang', 'mudah diakses', 'mudah digunakan', 'akses mudah', 'navigasi mudah', 'mudahan', 'mudh'],
    'Cepat': ['cepat', 'kilat', 'segera', 'fast', 'cepat mudah', 'cepat tanggap', 'cepat responsif', 'pantas', 'cpt', 'cepat merespon', 'cepat diselesaikan', 'respon cepat', 'penyelesaian cepat', 'cepatan', 'cpat'],
    'Baik': ['baik', 'bagus', 'prima', 'optimal', 'baik sesuai', 'sesuai dengan yang baik', 'kualitas baik', 'kinerja baik', 'good', 'bgus', 'bagus', 'kualitas tinggi', 'layanan baik', 'baikan', 'mantap', 'sesuai', 'cocok', 'tepat', 'pas', 'sinkron'],
    'Dipahami': ['dipahami', 'dimengerti', 'dipelajari', 'dipahami', 'understandable', 'mudah dipahami', 'sederhana dipahami', 'jelas dimengerti', 'easy to understand', 'mdh dimengerti', 'easily understood', 'mudah dimengerti', 'jelas dipahami', 'terstruktur dengan baik', 'instruksi yang jelas', 'mudah dipaham'],
    'Efisien': ['efisien', 'efficient', 'efektif', 'hemat', 'produktif', 'efis', 'hemat waktu', 'efisien tinggi', 'waktu penggunaan yang efisien', 'penggunaan sumber daya yang hemat', 'efisiennya', 'tepat waktu hasil', 'hasil yang tepat waktu'],
    'Meningkatkan': ['Meningkatkan', 'enhance', 'peningkatan', 'pembaharuan', 'menaikkan', 'memperbaiki', 'meningkat', 'improve', 'meningkatkan produktivitas', 'meningkatkan efisiensi', 'meningkatkan efektivitas', 'meningkatkan kualitas', 'mgkat'],
    'Stabil': ['stabil', 'tetap', 'kokoh'],
    'Pengguna': ['pengguna', 'penggunaan', 'tampilan', 'penampilan'],
    'Hasil': ['hasil', 'output', 'hasil produksi', 'hasil kerja', 'output', 'hsl', 'hasil akhir', 'output kerja', 'hasil yang diharapkan', 'hasil yang diinginkan', 'hasil-hasil'],
    'Kepuasan': ['kepuasan', 'satisfaction', 'satisfying', 'memenuhi harapan', 'kepuasan', 'kepuasan pengguna', 'kepuasan pelanggan', 'memuaskan', 'memuaskan hati', 'memberikan kepuasan', 'mgkatkan ksnsn', 'memadai', 'meningkatkan puas', 'Meningkatkan kepuasan', 'meningkatkan kepuasan pelanggan', 'meningkatkan kualitas layanan', 'memuaskan pelanggan', 'memberikan kepuasan', 'meningkatkan loyalitas', 'meningkatkan retensi pelanggan', 'fitur', 'fiturnya', 'meningkatkan kepuasan-kepuasan', 'pengalaman pengguna yang memuaskan', 'layanan yang memuaskan', 'kepuasan pelanggan yang berkelanjutan', 'kepuasan pengguna yang tinggi', 'memuaskan-memuaskan', 'memuaskn', 'mgkatkan ksnsn', 'msksn', 'memenuhi', 'puaskan', 'kegembiraan', 'kesenangan', 'kepuasan hati']
}

# Fungsi utama untuk menjalankan seluruh sistem
def main(app_id, count=1000, scale=5):
    # Fetch reviews
    df = fetch_reviews(app_id, count)

    # Process the reviews
    process_column(df, 'content')

    # Find similar reviews
    similar_reviews = find_similar_reviews(df['content'], keywords_similarity, threshold=0.09)

    # Convert to DataFrame
    similar_reviews_df = pd.DataFrame(similar_reviews, columns=['index', 'content_clean', 'similarity_score'])

    # Add original reviews for reference
    similar_reviews_df['content'] = df.loc[similar_reviews_df['index'] - 1, 'content'].values
    similar_reviews_df['userName'] = df.loc[similar_reviews_df['index'] - 1, 'userName'].values
    similar_reviews_df['score'] = df.loc[similar_reviews_df['index'] - 1, 'score'].values

    # Perform sentiment analysis and convert to Likert scale
    similar_reviews_df['sentiment_score'] = similar_reviews_df['content_clean'].apply(sentiment_analysis_vader)
    similar_reviews_df['sentiment_label'] = similar_reviews_df['sentiment_score'].apply(lambda x: sentiment_to_likert(x, scale))

    # Display DataFrame as table
    display(similar_reviews_df[['content', 'sentiment_score', 'sentiment_label']])

# Contoh penggunaan fungsi utama
app_id = 'id.or.muhammadiyah.quran'
count = 1000  # Jumlah review yang ingin diambil
scale = 5  # Skala Likert 1-5

main(app_id, count, scale)


,content,sentiment_score,sentiment_label
0,aplikasi bagus lengkap fiturnya mudah mudah de...,0.5994,5
1,bagus banyak fitur yg sesuai dg hpt waktu shol...,0.0000,3
2,sangat mudah mudah paham saran menu dzikir dan...,0.0000,3
3,di menu iqra saran masuk ikan voice suara tiap...,0.0000,3
4,sangat bantu ajar paham agama baik,0.0000,3
...,...,...,...
59,masyaallah bagus bangett,0.0000,3
60,mantap moga berkah,0.0000,3
61,mantappp appnya,0.0000,3
62,masya allah bagus bgt aplikasi sangat manfaat,0.0000,3
